In [4]:
# !pip install ollama -q

import json
import ollama

LLM_MODEL = "llama3"

In [5]:
%run 3_reranking.ipynb

c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\nbformat\validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)
c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2927.26it/s]


Jumlah dokumen di collection: 5000
['telur', 'dada ayam', 'bawang putih']
{'matched': 3, 'coverage_recipe': 0.6, 'coverage_user': 1.0, 'efficiency_score': 0.7499999999999999, 'matched_items': ['telur ayam', 'dada ayam fillet', 'bawang putih'], 'missing_items': ['kecap manis', 'garam']}
Bahan user (setelah parsing): ['telur', 'dada ayam', 'bawang putih']
[1] Telur Dadar Daun Bawang
    final_score=0.6250  (sbert_norm=1.000, efficiency=0.250)
    bahan resep: ['telur ayam', 'daun bawang merah', 'cabai', 'garam', 'merica bubuk']
    -> sudah dimiliki user: ['telur ayam']
    -> perlu ditambah: ['daun bawang merah', 'cabai', 'garam', 'merica bubuk']

[2] Telur Dadar
    final_score=0.5026  (sbert_norm=0.561, efficiency=0.444)
    bahan resep: ['telur', 'bawang merah', 'bawang putih', 'daun bawang', 'garam dan penyedap', 'minyak goreng']
    -> sudah dimiliki user: ['telur', 'bawang putih']
    -> perlu ditambah: ['bawang merah', 'daun bawang', 'garam dan penyedap', 'minyak goreng']

[3] Te

## Daftar Bahan Pelengkap

Daftar bahan-bahan yang **sering dipakai tapi jarang diketik user** karena dianggap "pasti ada". Lalu cek mana dari daftar ini yang belum disebut user, lalu itu jadi bahan kandidat untuk ditanyakan.

In [6]:
PANTRY_STAPLES = [
    "garam",
    "gula",
    "gula pasir",
    "merica",
    "merica bubuk",
    "lada",
    "minyak goreng",
    "kecap manis",
    "kecap asin",
    "saus tiram",
    "bawang merah",
    "bawang putih",
    "daun bawang",
    "cabai",
    "cabe rawit",
    "penyedap rasa",
    "kaldu bubuk",
    "air",
    "tepung terigu",
    "tepung maizena",
    "daun salam",
    "daun jeruk",
    "jahe",
    "kunyit",
    "lengkuas",
    "ketumbar",
]


def ingredient_matches(user_name: str, ref_name: str) -> bool:
    u, r = user_name.lower().strip(), ref_name.lower().strip()
    return bool(u) and bool(r) and (u in r or r in u)


def get_unmentioned_staples(user_ingredients: list) -> list:
    """Bahan dasar dapur yang belum disinggung user sama sekali."""
    unmentioned = []
    for staple in PANTRY_STAPLES:
        if not any(ingredient_matches(u, staple) for u in user_ingredients):
            unmentioned.append(staple)
    return unmentioned


# test
contoh_user = ["telur", "dada ayam", "bawang putih"]
print(get_unmentioned_staples(contoh_user))

['garam', 'gula', 'gula pasir', 'merica', 'merica bubuk', 'lada', 'minyak goreng', 'kecap manis', 'kecap asin', 'saus tiram', 'bawang merah', 'daun bawang', 'cabai', 'cabe rawit', 'penyedap rasa', 'kaldu bubuk', 'air', 'tepung terigu', 'tepung maizena', 'daun salam', 'daun jeruk', 'jahe', 'kunyit', 'lengkuas', 'ketumbar']


## Pertanyaan Konfirmasi

In [7]:
def generate_clarifying_question(
    user_text: str, user_ingredients: list, unmentioned_staples: list
) -> str:
    """Minta Llama3 membuat 1 pertanyaan konfirmasi yang natural dalam Bahasa Indonesia,
    tentang apakah user juga punya bahan-bahan dapur umum di `unmentioned_staples`.
    Tidak template -- LLM bebas milih gaya/urutan/kalimat, TIDAK boleh menambah bahan
    di luar daftar yang diberikan."""

    if not unmentioned_staples:
        return None  # semua bahan dasar udah disebut user, gak perlu nanya lagi

    system_prompt = (
        "Kamu adalah asisten resep masakan yang ramah dan ngobrol santai dalam Bahasa Indonesia. "
        "Tugasmu: tanyakan ke user apakah mereka juga punya sebagian dari daftar bahan dapur umum "
        "berikut, supaya rekomendasi resep bisa lebih pas. JANGAN menyebutkan semua bahan dalam daftar "
        "sekaligus seperti daftar belanja, pilih beberapa yang paling masuk akal/relevan dengan bahan "
        "yang sudah disebut user, dan tanyakan dengan kalimat yang mengalir, bukan format list. "
        "Jangan pernah menyebut bahan di luar daftar yang diberikan. Balas HANYA dengan pertanyaannya "
        "saja (1-2 kalimat), tanpa basa-basi pembuka seperti 'Tentu,' atau 'Baik,'."
    )
    user_prompt = (
        f"Bahan yang sudah disebut user: {', '.join(user_ingredients)}.\n"
        f'Input asli user: "{user_text}"\n'
        f"Daftar bahan dapur umum yang belum disebut: {', '.join(unmentioned_staples)}."
    )

    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        options={"temperature": 0.9},
    )
    return response["message"]["content"].strip()


# test
user_text = "2 butir telur, 100gr dada ayam, 2 siung bawang putih"
user_ingredients = ["telur", "dada ayam", "bawang putih"]
unmentioned = get_unmentioned_staples(user_ingredients)

question = generate_clarifying_question(user_text, user_ingredients, unmentioned)
print(question)

Kamu punya garam dan merica di rumah?


## Ekstrak Jawaban User Jadi Bahan Terstruktur

Llama3 mengekstrak itu jadi JSON, **dibatasi hanya ke bahan yang memang ada di `unmentioned_staples`** (biar tidak halusinasi nambahin bahan yang tidak pernah disebut user).

In [8]:
def parse_user_confirmation(user_reply: str, unmentioned_staples: list) -> list:
    """Ekstrak bahan yang dikonfirmasi user DARI daftar unmentioned_staples saja.
    Return: list nama bahan (subset dari unmentioned_staples)."""

    system_prompt = (
        "Kamu mengekstrak informasi dari jawaban user tentang bahan masakan yang mereka punya. "
        "Kamu HANYA boleh memilih dari daftar kandidat yang diberikan -- jangan menambah bahan lain, "
        "jangan menerka. Kalau user bilang tidak ada tambahan / cuma itu saja / dsb, kembalikan list kosong. "
        'Balas HANYA dengan JSON valid, format persis: {"confirmed_ingredients": ["...", ...]}. '
        "Tanpa penjelasan atau teks lain di luar JSON."
    )
    user_prompt = (
        f"Daftar kandidat bahan: {json.dumps(unmentioned_staples, ensure_ascii=False)}\n"
        f'Jawaban user: "{user_reply}"'
    )

    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        format="json",  # paksa Ollama keluarin JSON valid (didukung Llama3 di Ollama)
        options={"temperature": 0.2},  # rendah -- ini tugas ekstraksi, bukan kreatif
    )

    try:
        parsed = json.loads(response["message"]["content"])
        confirmed = parsed.get("confirmed_ingredients", [])
    except (json.JSONDecodeError, AttributeError):
        confirmed = []

    # safety net: filter ulang, pastikan hasil LLM benar2 subset dari kandidat (anti-halusinasi)
    confirmed = [
        c
        for c in confirmed
        if any(ingredient_matches(c, cand) for cand in unmentioned_staples)
    ]
    return confirmed


# test
user_reply = "aku ada garam sama minyak goreng doang kayaknya"
confirmed = parse_user_confirmation(user_reply, unmentioned)
print(confirmed)

['garam', 'minyak goreng']


## Gabung Bahan & Rerank Ulang

Bahan yang dikonfirmasi digabung ke `user_ingredients`, lalu panggil ulang `rerank()`, retrieve SBERT ulang + hitung ulang efficiency score dari awal.

In [9]:
def rerank_with_confirmation(
    user_text: str,
    user_ingredients: list,
    confirmed_extra: list,
    top_k_candidates: int = 30,
    top_n_final: int = 5,
):
    """Gabungkan bahan awal + bahan tambahan yang dikonfirmasi, lalu rerank ulang total.
    Catatan: rerank() dari notebook 03 mem-parsing ulang user_text mentah -- karena bahan
    tambahan tidak ada di user_text asli, kita panggil versi internalnya langsung supaya
    tidak perlu 'mengarang' ulang kalimat user."""

    final_user_ingredients = list(
        dict.fromkeys(user_ingredients + confirmed_extra)
    )  # gabung, buang duplikat

    candidates = retrieve_candidates(user_text, top_k=top_k_candidates)

    import numpy as np

    sbert_scores = np.array([c["sbert_score"] for c in candidates])
    s_min, s_max = sbert_scores.min(), sbert_scores.max()
    sbert_norm = (sbert_scores - s_min) / (s_max - s_min + 1e-9)

    for c, s_norm in zip(candidates, sbert_norm):
        eff = compute_efficiency_score(final_user_ingredients, c["ingredient_names"])
        c.update(eff)
        c["sbert_score_norm"] = float(s_norm)
        c["final_score"] = 0.5 * float(s_norm) + 0.5 * eff["efficiency_score"]

    candidates_sorted = sorted(candidates, key=lambda c: c["final_score"], reverse=True)
    return candidates_sorted[:top_n_final], final_user_ingredients


# test
final_results, final_user_ingredients = rerank_with_confirmation(
    user_text, user_ingredients, confirmed
)
print("Bahan final:", final_user_ingredients)
for c in final_results:
    print(c["title"], round(c["final_score"], 3))

Bahan final: ['telur', 'dada ayam', 'bawang putih', 'garam', 'minyak goreng']
Telur Dadar Daun Bawang 0.7
Telur Dadar 0.644
Telur Dadar Tempe 0.584
Telur Dadar Tempe 0.545
Telur Dadar 0.538


## Jawaban Akhir

In [10]:
def generate_recipe_blurb(fact: dict, user_ingredients: list) -> str:
    """1 komentar natural singkat untuk SATU resep. Dipanggil per-resep (bukan borongan)
    supaya konteks LLM kecil -> kecil kemungkinan skip resep / ketuker matched vs missing."""
    system_prompt = (
        "Kamu asisten resep masakan yang ngobrol natural dalam Bahasa Indonesia. "
        "Kamu akan diberi SATU resep (judul, bahan yang SUDAH dipunya user, bahan yang PERLU ditambah). "
        "Tulis 1-2 kalimat komentar natural tentang resep ini -- boleh singgung kenapa cocok/kurang cocok, "
        "atau kasih tips singkat. ATURAN KETAT: jangan pernah menyebut bahan 'sudah ada' sebagai 'perlu "
        "ditambah' atau sebaliknya -- gunakan PERSIS seperti yang diberi. Jangan mengarang bahan atau "
        "langkah masak yang tidak ada di data. Balas HANYA komentarnya, tanpa nomor urut atau judul resep "
        "(judul akan ditampilkan terpisah oleh sistem)."
    )
    user_prompt = (
        f"Bahan yang dipunya user: {', '.join(user_ingredients)}.\n"
        f"Resep: {json.dumps(fact, ensure_ascii=False)}"
    )
    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        options={"temperature": 0.8},
    )
    return response["message"]["content"].strip()


def generate_final_response(final_results: list, final_user_ingredients: list) -> str:
    """FIX: sebelumnya 1 panggilan LLM untuk narasikan SEMUA resep sekaligus -> kadang
    ada resep yang di-skip diam-diam, dan matched_items/missing_items bisa ketuker.
    Sekarang: judul + bahan kurang dicetak DETERMINISTIK oleh kode (dijamin akurat),
    LLM cuma nambahin komentar singkat per resep (dipanggil satu-per-satu)."""
    lines = []
    for i, c in enumerate(final_results, start=1):
        fact = {
            "judul": c["title"],
            "bahan_sudah_ada": c["matched_items"],
            "bahan_perlu_ditambah": c["missing_items"],
        }
        blurb = generate_recipe_blurb(fact, final_user_ingredients)
        kurang = (
            "tidak perlu tambahan apa pun -- bahannya lengkap!"
            if not c["missing_items"]
            else "perlu tambah: " + ", ".join(c["missing_items"])
        )
        lines.append(f"{i}. **{c['title']}** -- {kurang}\n   {blurb}")
    return "\n\n".join(lines)


final_message = generate_final_response(final_results, final_user_ingredients)
print(final_message)

1. **Telur Dadar Daun Bawang** -- perlu tambah: daun bawang merah, cabai, merica bubuk
   Resep ini sepertinya sangat cocok untuk penggemar telur dadar yang mencari variasi rasa dan aroma. Dengan penambahan daun bawang merah, resep ini akan memberikan sensasi fresh dan aroma yang kuat.

2. **Telur Dadar** -- perlu tambah: bawang merah, daun bawang
   Cerita lain sih, Telur Dadar ini rasanya makin lezat kalau kita tambahkan bawang merah yang kriuk-kriuk di dalamnya! Dan ya, daun bawang juga menambahkan aroma yang segar dan enak.

3. **Telur Dadar Tempe** -- perlu tambah: tempe, cabe rawit merah, daun jeruk
   Resep ini memang unik dan menarik! Penyebutan "Telur Dadar Tempe" memang membuat saya penasaran dengan cara membuatnya. Saya pikir, dengan bahan-bahan yang sudah ada seperti telur, bawang putih, dan garam, resep ini pasti memiliki rasa yang gurih dan pedas dari bawang putih dan cabe rawit merah.

4. **Telur Dadar Tempe** -- perlu tambah: tempe, daun bawang, cabe rawit besar, kaldu 

In [11]:
def find_closest_makeable_recipes(user_ingredients: list, top_n: int = 3) -> list:
    """Scan SEMUA resep di collection (bukan cuma hasil retrieval semantic), hitung efficiency
    score tiap resep, lalu urutkan berdasar JUMLAH bahan tambahan paling sedikit (bukan skor
    gabungan sbert+efficiency seperti di rerank() utama). 100% data asli dari database."""
    all_data = collection.get(
        include=["metadatas"]
    )  # ambil semua dokumen, tanpa query vector

    scored = []
    for rid, meta in zip(all_data["ids"], all_data["metadatas"]):
        ingredient_names = json.loads(meta["ingredient_names"])
        eff = compute_efficiency_score(user_ingredients, ingredient_names)
        scored.append(
            {
                "id": rid,
                "title": meta["title"],
                "ingredient_names": ingredient_names,
                **eff,
            }
        )

    # urutkan: jumlah bahan yg perlu ditambah paling sedikit dulu, lalu coverage_user paling tinggi
    # (supaya di antara yang sama2 'kurang 2 bahan', yang paling banyak makein bahan user menang)
    scored_sorted = sorted(
        scored, key=lambda c: (len(c["missing_items"]), -c["coverage_user"])
    )
    return scored_sorted[:top_n]


def get_alternative_recipe_section(final_user_ingredients: list) -> str:
    """Bagian 'resep paling pas dgn bahan yang ada' -- selalu dari database, tidak pernah generatif."""
    closest = find_closest_makeable_recipes(final_user_ingredients, top_n=3)
    if not closest:
        return "Belum nemu resep yang cocok di database untuk bahan-bahan ini."

    lines = []
    for c in closest:
        if not c["missing_items"]:
            status = "bahannya udah 100% kamu punya, langsung bisa dimasak!"
        else:
            status = f"cuma kurang {len(c['missing_items'])} bahan: {', '.join(c['missing_items'])}"
        lines.append(f"- **{c['title']}** -- {status}")
    return (
        "Resep paling pas dengan bahan yang kamu punya (dari seluruh database):\n"
        + "\n".join(lines)
    )


# test
print(get_alternative_recipe_section(final_user_ingredients))

Resep paling pas dengan bahan yang kamu punya (dari seluruh database):
- **Telur merah dadar pas pedasnya dan simpel** -- bahannya udah 100% kamu punya, langsung bisa dimasak!
- **Telor Bebek Ceplok** -- bahannya udah 100% kamu punya, langsung bisa dimasak!
- **Telur asin** -- bahannya udah 100% kamu punya, langsung bisa dimasak!


## Alur

In [12]:
def chatbot_flow(user_text: str, user_reply_to_confirmation: str):
    # 1) parsing bahan awal
    user_ingredients = parse_user_ingredients(user_text)

    # 2) cek bahan dasar dapur yang belum disebut
    unmentioned = get_unmentioned_staples(user_ingredients)

    # 3) LLM susun pertanyaan konfirmasi (kalau ada yang perlu ditanyakan)
    question = generate_clarifying_question(user_text, user_ingredients, unmentioned)
    if question:
        print("BOT:", question)
        print("USER:", user_reply_to_confirmation)

    # 4) ekstrak jawaban user
    confirmed_extra = (
        parse_user_confirmation(user_reply_to_confirmation, unmentioned)
        if question
        else []
    )

    # 5) rerank ulang
    final_results, final_user_ingredients = rerank_with_confirmation(
        user_text, user_ingredients, confirmed_extra
    )

    # 6) jawaban akhir
    final_message = generate_final_response(final_results, final_user_ingredients)
    print("\nBOT:", final_message)

    # 6b) resep paling pas
    alt_section = get_alternative_recipe_section(final_user_ingredients)
    print("\nBOT (alternatif):", alt_section)

    return final_results, final_user_ingredients, final_message, alt_section


# contoh
chatbot_flow(
    user_text="2 butir telur, 100gr dada ayam, 2 siung bawang putih",
    user_reply_to_confirmation="ada garam sama minyak goreng doang kayaknya, sisanya gak ada",
)

BOT: Apakah Anda punya garam dan merica di rumah, karena biasanya digunakan sebagai penyedap rasa pada masakan?
USER: ada garam sama minyak goreng doang kayaknya, sisanya gak ada

BOT: 1. **Telur Dadar Daun Bawang** -- perlu tambah: daun bawang merah, cabai, merica bubuk
   Resep Telur Dadar Daun Bawang ini terlihat sangat menarik! Yang bikin kagum adalah penambahan daun bawang merah yang akan memberikan rasa pedas dan aroma yang kuat. Saya pikir cabai juga akan menambahkan sensasi pedas yang pas, tapi mungkin perlu diihatkan kembali kekentalannya agar tidak terlalu pedas.

2. **Telur Dadar** -- perlu tambah: bawang merah, daun bawang
   Resep Telur Dadar ini sangat cocok untuk dijadikan sarapan atau makanan ringan! Penambahan bawang merah dan daun bawang akan memberikan rasa dan tekstur yang lebih berani dan segar.

3. **Telur Dadar Tempe** -- perlu tambah: tempe, cabe rawit merah, daun jeruk
   Resep ini terlihat sangat menarik, apalagi dengan penambahan tempe yang memberikan tekstur

([{'id': 'resep_2582',
   'title': 'Telur Dadar Daun Bawang',
   'ingredient_names': ['telur ayam',
    'daun bawang merah',
    'cabai',
    'garam',
    'merica bubuk'],
   'steps_text': '1. Pisahkan Bawang merah dengan daunnya. Iris tipis bawang merah. Iris anjang daun bawang. Rajang cabe. 2. Kocok telur bersama garam, merica, kemudian masukkan daun bawang, bawang merah dna cabai. 3. Panaskan minyak di wajan, tuang adonan telur. Masak dengan api sedang hingga kedua sisi matang.',
   'sbert_score': 0.7709106206893921,
   'matched': 2,
   'coverage_recipe': 0.4,
   'coverage_user': 0.4,
   'efficiency_score': 0.4000000000000001,
   'matched_items': ['telur ayam', 'garam'],
   'missing_items': ['daun bawang merah', 'cabai', 'merica bubuk'],
   'sbert_score_norm': 0.9999999915642659,
   'final_score': 0.699999995782133},
  {'id': 'resep_4346',
   'title': 'Telur Dadar',
   'ingredient_names': ['telur',
    'bawang merah',
    'bawang putih',
    'daun bawang',
    'garam dan penyedap',
